In [1]:
import pandas as pd
import os
import re
from gensim.models import FastText, Word2Vec
from collections import defaultdict
from collections import Counter
from itertools import chain
import numpy as np
from scipy.interpolate import make_interp_spline
import matplotlib as plt
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from collections import defaultdict, Counter
from itertools import chain
from gensim.models import FastText
from scipy.interpolate import make_interp_spline

In [2]:
ft_htr    = FastText.load("../../../data/H8/word-embeddings/FastText_htr.model")

In [3]:
kernwoorden = [
    'lawijd', 'lawaai', 'gerucht', 'kabaal', 'rumoer',
    'nachtlawijd', 'nachtlawaai', 'nachtrumoer', 'nachtgerucht', 'nachtrust'
]

variantengroepen = defaultdict(set)

for kernwoord in kernwoorden:
    variantengroepen[kernwoord].add(kernwoord)

    if kernwoord in ft_htr.wv:
        for woord, score in ft_htr.wv.most_similar(kernwoord, topn=300):
            freq = ft_htr.wv.get_vecattr(woord, 'count')

            if score >= 0.75 and freq >= 1:
                if woord.startswith('nacht') and not kernwoord.startswith('nacht'):
                    continue
                if not woord.startswith('nacht') and kernwoord.startswith('nacht'):
                    continue

                variantengroepen[kernwoord].add(woord)

alle_varianten = set().union(*variantengroepen.values())

print(f"Totaal unieke varianten: {len(alle_varianten)}")

Totaal unieke varianten: 695


In [4]:
rijen = []

for kernwoord, varianten in variantengroepen.items():
    for variant in sorted(varianten):
        rijen.append({
            'kernwoord': kernwoord,
            'variant': variant
        })

df_varianten = pd.DataFrame(rijen)

df_varianten

,kernwoord,variant
0,lawijd,achtlawij
1,lawijd,aldaarluidkeels
2,lawijd,awij
3,lawijd,bawijd
4,lawijd,bawijt
...,...,...
1653,nachtrust,nachtruust
1654,nachtrust,nachtruuwe
1655,nachtrust,nachtruuwer
1656,nachtrust,nachtruwver


In [5]:
varianten = '../../../data/H8/word-embeddings/fasttext_varianten.csv'

df_varianten.to_csv(
    varianten,
    index=False
)

print(varianten)

../../../data/H8/word-embeddings/fasttext_varianten.csv


In [6]:
def zoek_geluidsoverlast(tekst, varianten):
    if not isinstance(tekst, str):
        return ''
    woorden = tekst.lower().split()
    gevonden = []
    for woord in woorden:
        woord_schoon = re.sub(r'[^\w]', '', woord)
        if woord_schoon in varianten and woord_schoon not in gevonden:
            gevonden.append(woord_schoon)
    return '; '.join(gevonden) if gevonden else ''

In [7]:
map_tekst = '../../../data/algemeen/gebeurtenisdata_macro'

bestanden = [
    f for f in os.listdir(map_tekst)
    if f.endswith('.xlsx')
    and not f.startswith('~$')
    and not f.startswith('._')
]

alle_matches = []

for bestand in bestanden:
    pad = os.path.join(map_tekst, bestand)

    try:
        df = pd.read_excel(pad)

        df['geluidsoverlast'] = df['incident_text'].apply(
            lambda tekst: zoek_geluidsoverlast(tekst, alle_varianten)
        )

        df['bronbestand'] = bestand

        df_match = df[df['geluidsoverlast'] != ''].copy()
        alle_matches.append(df_match)

    except Exception:
        continue

df_totaal = pd.concat(alle_matches, ignore_index=True)

uitvoer = '../../../data/H8/word-embeddings/geluidsoverlast_fasttextvarianten.xlsx'
df_totaal.to_excel(uitvoer, index=False)

In [8]:
df_totaal.to_excel(uitvoer, index=False)

print(f"Opgeslagen: {Path(uitvoer).resolve()}")

Opgeslagen: /Volumes/Extreme SSD/doctoraat/FINAAL_poefschrift/X. Bijlagen copy/Data en code/data/H8/word-embeddings/geluidsoverlast_fasttextvarianten.xlsx


In [9]:
groepen_samengevoegd = {
    'lawijd': set(df_varianten.loc[
        df_varianten['kernwoord'].isin(['lawijd', 'nachtlawijd']), 'variant'
    ]),
    'lawaai': set(df_varianten.loc[
        df_varianten['kernwoord'].isin(['lawaai', 'nachtlawaai']), 'variant'
    ]),
    'rumoer': set(df_varianten.loc[
        df_varianten['kernwoord'].isin(['rumoer', 'nachtrumoer']), 'variant'
    ]),
    'kabaal': set(df_varianten.loc[
        df_varianten['kernwoord'] == 'kabaal', 'variant'
    ]),
    'gerucht': set(df_varianten.loc[
        df_varianten['kernwoord'].isin(['gerucht', 'nachtgerucht']), 'variant'
    ]),
    'nachtrust': set(df_varianten.loc[
        df_varianten['kernwoord'] == 'nachtrust', 'variant'
    ])
}

{k: len(v) for k, v in groepen_samengevoegd.items()}

{'lawijd': 367,
 'lawaai': 439,
 'rumoer': 355,
 'kabaal': 40,
 'gerucht': 340,
 'nachtrust': 117}